# Phonon frequencies from frozen-phonon SCFs (Y₂Ir₂O₇ Γ₂⁻)

Computes the frequencies and eigenvectors of the axion-active Y₂Ir₂O₇ Γ₂⁻ modes from pw.x SCFs of
displaced structures. Set the runs in the settings cell. Everything is a NumPy array:

| array | shape | meaning |
|---|---|---|
| `R0`, `F0` | (nat, 3) | base positions (Å) and forces (eV/Å) |
| `R`, `F` | (nrun, nat, 3) | the same for every displaced run |
| `E` | (nrun,) | total energy minus the base energy (eV) |
| `V` | (nb, nat, 3) | the symmetry-adapted (SMODES) basis vectors, entries ±1 |
| `M` | (nb, nb) | mass matrix in that basis (amu) |
| `Phi` | (nb, nb) | force-constant matrix in that basis (eV/Å²) |

**The idea.** A displacement Δr = Σᵢ Qᵢ vᵢ inside one irrep costs energy ½ QᵀΦQ and has kinetic energy ½ Q̇ᵀMQ̇, with

$$\Phi_{ij} = \frac{\partial^2 E}{\partial Q_i\,\partial Q_j} = -\,\mathbf v_i\cdot\frac{\partial \mathbf F}{\partial Q_j},\qquad
M_{ij} = \sum_a m_a\,\mathbf v_{i,a}\cdot\mathbf v_{j,a}.$$

The normal modes solve $\Phi\,\mathbf c = \omega^2 M\,\mathbf c$. Symmetry makes Φ block-diagonal by irrep and the
force constants are time-even, so the nb × nb block of one irrep is exact: no other irrep mixes in.

**Four ways of getting at Φ, all used below:**
1. **Forces of the basis runs** (one SCF per basis vector, amplitude `Q_BASIS`): the whole matrix, $\Phi_{ij} = -\mathbf v_i\cdot[\mathbf F(Q_j) - \mathbf F(0)]/Q$.
2. **Energies of the basis runs:** the diagonal only, $\Phi_{ii} = 2\,\Delta E_i / Q^2$. Compared with method 1 it tests the forces.
3. **A run along another pattern c** (e.g. the rigid septuple-layer motion, larger amplitude, tighter conv_thr):
   its energy gives $\mathbf c^T\Phi\,\mathbf c = 2\Delta E/Q^2$ and its forces give the whole row $\Phi\mathbf c$.
4. **Combined:** method 1's matrix with its c row/column replaced by method 3's. A cross-check only.

**Production is method 1** (paper protocol, 2026-10-15), accepted when: E/F on the diagonal within 1%, Φ symmetric
within 1%, SCF force correction below 1% of the force change. Methods 2-4 are the checks.

Notes: [`notes/log/2026-10-14_mbt_t2m_dynmat.md`](../../notes/log/2026-10-14_mbt_t2m_dynmat.md),
[`notes/log/2026-10-15_mbt_rigid_layer_mode.md`](../../notes/log/2026-10-15_mbt_rigid_layer_mode.md),
YIO: [`notes/log/2026-10-12_yio_lda_modes.md`](../../notes/log/2026-10-12_yio_lda_modes.md) §16.

In [1]:
import re
from pathlib import Path

import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "axion_phonon/axion.py").exists())
DATA = ROOT / "data"

RY_EV = 13.605693                 # Ry -> eV
BOHR_A = 0.529177                 # bohr -> Å
CM_PER_SQRT = np.sqrt(1.602176634e-19 / 1e-20 / 1.66053907e-27) / (2 * np.pi * 2.99792458e10)  # sqrt(eV/Å²/amu) -> cm^-1

## Settings

`BASIS_RUNS[i]` must be the SCF displaced by `Q_BASIS` × (SMODES vector i). `EXTRA_RUNS` are SCFs along any other
pattern in the same irrep, with their amplitudes in `EXTRA_Q`; they must all be along the same pattern. Leave
`EXTRA_RUNS = []` to use the basis runs only. The commented block is the Y₂Ir₂O₇ Γ₂⁻ setting.

In [2]:
PREFIX = "Y2Ir2O7"
BASE = DATA / "Y2Ir2O7/base/lda/scf_conv1e-10_213998"                # base SCF (conv_thr 1e-10)
BASIS_RUNS = [DATA / "Y2Ir2O7/phonon/GM2-/mode1/Q_0.01A/lda/scf_nscf_conv1e-10_213999",   # Y
              DATA / "Y2Ir2O7/phonon/GM2-/mode2/Q_0.01A/lda/scf_nscf_conv1e-10_214003",   # Ir
              DATA / "Y2Ir2O7/phonon/GM2-/mode3/Q_0.01A/lda/scf_nscf_conv1e-10_214016"]   # O48f
BASIS_NAMES = ["Y", "Ir", "O"]
Q_BASIS = 0.01                                                      # Å
EXTRA_RUNS, EXTRA_Q = [], []
MASS = {"Y": 88.90584, "Ir": 192.217, "O": 15.9994}                 # amu, keyed by species without digits
DTHETA_DQ = [-0.017145, 0.054737, 0.063766]                         # rad/Å per basis vector: production (Q -> 0,
                                                                    # self-test base 214249, series windows, nk 10)


## Read the SCFs

Positions come from `<run>/<prefix>.scf.in` (`ATOMIC_POSITIONS angstrom`), energy and forces from
`<run>/<prefix>.scf.out`. The total forces are the first `nat` lines after the last "Forces acting on atoms". A
base run without `tprnfor` gets `F0 = 0`, which is harmless: see the check after the basis is built.

In [3]:
def read_scf(run):
    txt_in = (run / f"{PREFIX}.scf.in").read_text()
    blk = txt_in.split("ATOMIC_POSITIONS angstrom")[1].split("K_POINTS")[0]
    rows = [l.split() for l in blk.strip().splitlines() if l.strip()]
    species = [re.sub(r"\d", "", r[0]) for r in rows]
    pos = np.array([[float(x) for x in r[1:4]] for r in rows])
    txt = (run / f"{PREFIX}.scf.out").read_text()
    energy = float(re.findall(r"!\s+total energy\s+=\s+(\S+)", txt)[-1]) * RY_EV
    force, scf_corr = np.zeros_like(pos), np.nan
    if "Forces acting on atoms" in txt:
        f = txt.split("Forces acting on atoms")[-1]
        force = np.array(re.findall(r"atom\s+\d+ type\s+\d+\s+force =\s+(\S+)\s+(\S+)\s+(\S+)", f)[:len(pos)], float)
        force *= RY_EV / BOHR_A
        scf_corr = float(re.findall(r"Total SCF correction =\s+(\S+)", f)[0]) * RY_EV / BOHR_A
    return species, pos, force, energy, scf_corr

species, R0, F0, E_base, _ = read_scf(BASE)
m = np.array([MASS[s] for s in species])                       # (nat,)
runs = list(BASIS_RUNS) + list(EXTRA_RUNS)
data = [read_scf(r) for r in runs]
R = np.array([d[1] for d in data])                              # (nrun, nat, 3) Å
F = np.array([d[2] for d in data])                              # (nrun, nat, 3) eV/Å
E = np.array([d[3] for d in data]) - E_base                     # (nrun,) eV
scf_corr = np.array([d[4] for d in data])                       # (nrun,) eV/Å
nb, nat = len(BASIS_RUNS), len(R0)

print(f"{PREFIX}: {nat} atoms, base forces read: {bool(np.abs(F0).max() > 0)}, max |F0| = {np.linalg.norm(F0, axis=1).max():.3f} eV/Å")
print(f"{'run':40s} {'ΔE (meV)':>10s} {'max|Δr| (Å)':>12s} {'SCF corr (eV/Å)':>16s}")
for r, e, d, c in zip(runs, E, R - R0, scf_corr):
    print(f"{r.name:40s} {e*1e3:10.4f} {np.linalg.norm(d, axis=1).max():12.4f} {c:16.4f}")

Y2Ir2O7: 22 atoms, base forces read: True, max |F0| = 0.051 eV/Å
run                                        ΔE (meV)  max|Δr| (Å)  SCF corr (eV/Å)
scf_nscf_conv1e-10_213999                   21.9495       0.0173           0.0002
scf_nscf_conv1e-10_214003                   24.6173       0.0173           0.0005
scf_nscf_conv1e-10_214016                    7.5214       0.0100           0.0005


## Basis and mass matrix

The basis vectors are read back from the structures, $\mathbf v_i = (\mathbf R_i - \mathbf R_0)/Q$, so they are
exactly what was applied. The base's residual forces (MnBi₂Te₄ is the unrelaxed experimental cell) belong to the
fully symmetric irrep, so their projection on every basis vector is zero by symmetry. That is why subtracting
F(0) changes nothing in Φ, and why YIO's base needs no forces.

In [4]:
V = (R[:nb] - R0) / Q_BASIS                                     # (nb, nat, 3)
M = np.einsum("a,iax,jax->ij", m, V, V)                         # (nb, nb) amu
print("M (amu):\n", M.round(2))
print("v_i . F(0) (eV/Å, should be ~0):", np.einsum("iax,ax->i", V, F0))

M (amu):
 [[1066.87    0.      0.  ]
 [   0.   2306.6     0.  ]
 [   0.      0.    191.99]]
v_i . F(0) (eV/Å, should be ~0): [0.00000000e+00 0.00000000e+00 1.68356031e-14]


## Method 1: the full Φ from the forces of the basis runs

$\Phi_{ij} = -\mathbf v_i\cdot[\mathbf F(Q_j)-\mathbf F(0)]/Q$. The finite-difference Φ is not exactly symmetric;
its asymmetry is one measure of the force noise. We use the symmetric part. `scipy.linalg.eigh(Phi, M)` solves
$\Phi\mathbf c = \omega^2 M\mathbf c$ and returns eigenvectors with $\mathbf c^T M\mathbf c = 1$ amu. A negative ω²
is printed as a negative frequency.

In [5]:
dF = F - F0                                                     # (nrun, nat, 3)
Phi_raw = -np.einsum("iax,jax->ij", V, dF[:nb]) / Q_BASIS       # (nb, nb) eV/Å²; column j = run j
Phi1 = (Phi_raw + Phi_raw.T) / 2
print("Phi (eV/Å²):\n", Phi_raw.round(3))
print("asymmetry max|Phi - Phi^T| / max|Phi| =", round(np.abs(Phi_raw - Phi_raw.T).max() / np.abs(Phi_raw).max(), 4))

def to_cm(w2):
    return np.sign(w2) * np.sqrt(np.abs(w2)) * CM_PER_SQRT

def mode_table(Phi, label):
    w2, C = scipy.linalg.eigh(Phi, M)                           # C[:, n] is mode n, C^T M C = 1
    share = np.array([[np.sum((C[i, n] * V[i])**2) for i in range(nb)] for n in range(nb)])
    share /= share.sum(1, keepdims=True)
    print(f"\n{label}")
    print(f"{'ω (cm^-1)':>10s} {'meV':>7s} {'THz':>7s}   share of |Δr|² " + " / ".join(BASIS_NAMES))
    for n in range(nb):
        print(f"{to_cm(w2[n]):10.1f} {to_cm(w2[n])/8.06554:7.2f} {to_cm(w2[n])*0.0299792458:7.3f}   {share[n].round(3)}")
    return w2, C

w2_1, C_1 = mode_table(Phi1, "Method 1: forces of the basis runs")

Phi (eV/Å²):
 [[440.315  29.732  44.789]
 [ 29.716 492.47  -63.216]
 [ 44.75  -63.077 149.774]]
asymmetry max|Phi - Phi^T| / max|Phi| = 0.0003

Method 1: forces of the basis runs
 ω (cm^-1)     meV     THz   share of |Δr|² Y / Ir / O
     228.6   28.35   6.854   [0.042 0.667 0.29 ]
     328.3   40.70   9.842   [0.796 0.016 0.187]
     471.6   58.47  14.137   [0.01  0.002 0.988]


## Method 2: the diagonal of Φ from the energies of the basis runs

For a harmonic energy, $\Delta E_i = \tfrac12 \Phi_{ii} Q^2$ and $-\mathbf v_i\cdot\Delta\mathbf F_i = \Phi_{ii} Q$, so the
two estimates must agree. The ratio is the "E vs F" check in the notes. Energies converge quadratically in the
SCF error and forces only linearly, so when they disagree the energy is usually the better number. The energy
gives only the diagonal, though, so it cannot replace method 1.

In [6]:
Phi_E_diag = 2 * E[:nb] / Q_BASIS**2                           # (nb,)
print(f"{'basis':6s} {'Phi_ii from E':>14s} {'from F':>10s} {'E/F':>7s}   (eV/Å²)")
for name, pe, pf in zip(BASIS_NAMES, Phi_E_diag, np.diag(Phi_raw)):
    print(f"{name:6s} {pe:14.3f} {pf:10.3f} {pe/pf:7.3f}")

basis   Phi_ii from E     from F     E/F   (eV/Å²)
Y             438.990    440.315   0.997
Ir            492.346    492.470   1.000
O             150.427    149.774   1.004


## Method 3: a run along another pattern c

A run displaced along $\Delta\mathbf r = Q\sum_i c_i\mathbf v_i$ gives:
- from its energy, the curvature along c: $\mathbf c^T\Phi\,\mathbf c = 2\Delta E/Q^2$;
- from its forces, the whole row: $(\Phi\mathbf c)_i = -\mathbf v_i\cdot\Delta\mathbf F/Q$.

Two amplitudes test that E ∝ Q², with no linear term and no anharmonicity. The ratio $\mathbf c^T\Phi\mathbf c / \mathbf c^T M\mathbf c$ is
the Rayleigh quotient of c: the exact ω² when c is a normal mode, and in general a value between the lowest and
the highest ω². For MnBi₂Te₄, c = (1, −1, 1, −1) is the rigid antiphase motion of the two septuple layers, where
method 1 has the least precision: $\mathbf c^T\Phi\mathbf c$ is a ~2% remainder of entries of order 5-30 eV/Å².
It was wrong in sign with the Q = 0.01 Å, conv_thr 1e-7 basis runs; method 1 at Q = 0.03 Å, 1e-10 must reproduce it.

In [7]:
if EXTRA_RUNS:
    D = (R[nb:] - R0)                                           # (nex, nat, 3)
    Q_ex = np.array(EXTRA_Q)
    c_ex = np.linalg.lstsq(V.reshape(nb, -1).T, (D / Q_ex[:, None, None]).reshape(len(Q_ex), -1).T, rcond=None)[0].T  # (nex, nb)
    c = c_ex[0]
    assert np.allclose(c_ex, c, atol=1e-6), "EXTRA_RUNS must all be along the same pattern"
    k_E = 2 * E[nb:] / Q_ex**2                                  # (nex,) c^T Phi c from energy
    Phi_c = -np.einsum("iax,nax->ni", V, dF[nb:]) / Q_ex[:, None]  # (nex, nb) the row Phi c from forces
    k_F = Phi_c @ c
    cMc = c @ M @ c
    print("c =", c.round(4), f"  c^T M c = {cMc:.1f} amu")
    print(f"{'Q (Å)':>6s} {'ΔE (meV)':>9s} {'cΦc from E':>11s} {'from F':>8s} {'method 1':>9s}  ω_Rayleigh (cm^-1) E / F")
    for q, e, ke, kf in zip(Q_ex, E[nb:], k_E, k_F):
        print(f"{q:6.3f} {e*1e3:9.4f} {ke:11.4f} {kf:8.4f} {c @ Phi1 @ c:9.4f}  {to_cm(ke/cMc):6.1f} {to_cm(kf/cMc):6.1f}")
    print("\nPhi c from these forces:", Phi_c.round(3), "\nPhi c from method 1:   ", (Phi1 @ c).round(3))

    qq = np.linspace(0, Q_ex.max() * 1.1, 50)
    plt.figure(figsize=(4.5, 3.2))
    plt.plot(Q_ex, E[nb:] * 1e3, "o", label="SCF")
    plt.plot(qq, 0.5 * k_E.mean() * qq**2 * 1e3, "-", label=r"$\frac{1}{2}\,c^T\Phi c\,Q^2$")
    plt.xlabel("Q (Å per atom)"); plt.ylabel("E − E_base (meV)"); plt.legend(); plt.tight_layout(); plt.show()

## Method 4: method 1 with the c row and column replaced

Change to a basis whose first vector is c and whose other three are M-orthogonal to it:
$B = [\mathbf c, \mathbf b_2, \mathbf b_3, \mathbf b_4]$. In that basis the first row and column of $B^T\Phi B$ are
$B^T(\Phi\mathbf c)$, which method 3 measures. The remaining 3×3 block still comes from method 1. Transform back
and solve as before. This only checks method 1: if the two agree, method 1 needs no help along c.

In [8]:
if EXTRA_RUNS:
    B = np.eye(nb)
    B[:, 0] = c
    for j in range(1, nb):
        B[:, j] -= c * (c @ M @ B[:, j]) / cMc
    Pb = B.T @ Phi1 @ B
    Pb[0, :] = Pb[:, 0] = B.T @ Phi_c.mean(0)
    Binv = np.linalg.inv(B)
    Phi4 = Binv.T @ Pb @ Binv                                   # (nb, nb) eV/Å², SMODES basis
    print("Phi, method 4 (eV/Å²):\n", Phi4.round(3))
    w2_4, C_4 = mode_table(Phi4, "Method 4: method 1 + the rigid-pattern row")
    print("\noverlap of each mode with c:", np.abs(C_4.T @ M @ c / np.sqrt(cMc)).round(4))
Phi_final, w2_final, C_final = Phi1, w2_1, C_1                    # production: method 1; method 4 is the cross-check

## dθ per normal mode

With `DTHETA_DQ` = dθ/dQ for each basis vector (rad/Å, from `run_axion.py`), the normal-mode coordinate q
($\Delta\mathbf r = q\sum_i C_{in}\mathbf v_i$, $C^TMC = 1$ amu) has $d\theta/dq = \sum_i C_{in}\, d\theta/dQ_i$ in
rad/(Å·√amu). Dividing by the length of $\sum_i C_{in}\mathbf v_i$ gives dθ per Å of total displacement.

In [9]:
if DTHETA_DQ is not None:
    dth = np.array(DTHETA_DQ)                                   # (nb,)
    dth_q = dth @ C_final                                       # (nb,) rad/(Å √amu)
    length = np.linalg.norm(np.einsum("in,iax->nax", C_final, V).reshape(nb, -1), axis=1)
    print(f"{'ω (cm^-1)':>10s} {'dθ/dq (rad/(Å√amu))':>21s} {'dθ per Å of |Δr|':>17s}")
    for w2n, a, L in zip(w2_final, dth_q, length):
        print(f"{to_cm(w2n):10.1f} {a:+21.5f} {a/L:+17.5f}")
else:
    print("DTHETA_DQ not set")

 ω (cm^-1)   dθ/dq (rad/(Å√amu))  dθ per Å of |Δr|
     228.6              -0.00204          -0.02384
     328.3              +0.00118          +0.01037
     471.6              -0.00415          -0.01713


## Status

- **Y₂Ir₂O₇ Γ₂⁻ (LDA+U), method 1**, basis runs 213999 / 214003 / 214016 (Q = 0.01 Å, conv_thr 1e-10; the same runs
  as the amplitude series, `amplitude_series.ipynb`, which also extrapolates Φ to Q → 0).
- dθ/dQ per pattern: the production values (log 10-17, Q → 0).